# 02 — Fiabilidad y clases
*¿En qué podemos confiar?*

El modelo YOLOv5n se usa **sin fine-tuning** en el dominio de conducción y **sin ground-truth exhaustivo**. Este notebook convierte esa limitación en método: (A2) triaje de relevancia de clase, sensibilidad al umbral, y (A1) una validación manual *parcial* que produce precisión por clase con intervalos de confianza honestos (Wilson) y un umbral operativo justificado por clase.

> **Alcance del análisis de peligro:** Horn(0) + Siren(1). Speech(4) se trata como *contexto*, nunca como peligro. El resto son clases del dataset YOLO original (fuera de dominio).

In [ ]:
import sys, warnings
sys.path.insert(0, "../scripts")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geo_utils as gu

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.width", 140)

geo = gu.load_geo()
geo = geo[~geo["trayecto"].isin(gu.BAD_TRAYECTOS)].copy()
geo["class_name"] = geo["class"].map(gu.CLASS_NAMES)
print(f"{len(geo):,} detecciones geolocalizadas | fuentes: {geo['source'].unique().tolist()}")
geo.head(3)

## A2 — Triaje de relevancia de clase / desajuste de dominio
Cuántas detecciones aporta cada clase y qué fracción cae **fuera de dominio** (imposible/ruido en un coche). Argumento para restringir el peligro a Horn+Siren.

In [ ]:
counts = (geo.groupby("class")
          .size().rename("n").reset_index())
counts["name"] = counts["class"].map(gu.CLASS_NAMES)
def relevancia(c):
    if c in gu.ROAD_CLASSES:    return "Vial (peligro)"
    if c in gu.CONTEXT_CLASSES: return "Contexto"
    return "Fuera de dominio"
counts["grupo"] = counts["class"].apply(relevancia)
counts["pct"] = (100 * counts["n"] / counts["n"].sum()).round(1)
display(counts.sort_values("n", ascending=False))
print("Fuera de dominio:", counts.loc[counts.grupo=="Fuera de dominio","pct"].sum().round(1), "% de las detecciones")

In [ ]:
order = counts.sort_values("n", ascending=False)
palette = {"Vial (peligro)": "#c0392b", "Contexto": "#2980b9", "Fuera de dominio": "#95a5a6"}
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=order, x="name", y="n", hue="grupo", palette=palette, dodge=False, ax=ax)
ax.set_yscale("log"); ax.set_xlabel(""); ax.set_ylabel("detecciones (log)")
ax.set_title("Distribución de clases por relevancia de dominio")
plt.xticks(rotation=35, ha="right"); plt.tight_layout()
plt.savefig("../outputs/nbA_class_domain.png", dpi=130, bbox_inches="tight"); plt.show()

**Lectura.** Speech domina (radio/conversación). Horn y Siren —las únicas señales de peligro vial— son minoritarias (Siren especialmente escasa). Una fracción grande de detecciones pertenece a clases fuera de dominio: deben considerarse falsos positivos y excluirse del danger score.

## Sensibilidad al umbral de confianza
El umbral 0.10→0.80 mueve drásticamente el recuento. ¿Cuántos eventos sobreviven por clase y fuente a cada umbral?

In [ ]:
ths = np.round(np.arange(0.1, 0.96, 0.05), 2)
focus = [0, 1, 4]
rows = []
for th in ths:
    sub = geo[geo.confidence >= th]
    for c in focus:
        for src in ["mic", "mobile"]:
            rows.append({"th": th, "class": gu.CLASS_NAMES[c], "source": src,
                         "n": int(((sub["class"]==c) & (sub.source==src)).sum())})
sens = pd.DataFrame(rows)
g = sns.relplot(data=sens, x="th", y="n", hue="class", style="source",
                kind="line", marker="o", height=5, aspect=1.5)
g.set(xlabel="umbral de confianza", ylabel="detecciones")
g.fig.suptitle("Sensibilidad al umbral — Horn / Siren / Speech", y=1.02)
g.savefig("../outputs/nbA_threshold_sensitivity.png", dpi=130, bbox_inches="tight")

## A1 — Validación manual parcial + calibración
Sin ground-truth completo. Tomamos una **muestra estratificada** (clase × bin de confianza), la escuchamos y etiquetamos TP/FP. Con ~50–80 etiquetas estimamos precisión por clase con **IC de Wilson** (intervalos anchos, declarados).

**Paso 1 — exportar hoja de muestreo** (`validation/sampling_sheet.csv`). Rellenar columna `is_tp` ∈ {0,1} escuchando cada clip (`source_file` + `t_start`/`t_end`).

In [ ]:
from pathlib import Path
Path("../validation").mkdir(exist_ok=True)
N_PER_CELL = 8          # detecciones por celda clase×bin
TARGET_CLASSES = [0, 1, 4, 5, 6]   # viales + contexto + 2 fuera-dominio de control
bins = pd.IntervalIndex.from_breaks([0.1, 0.3, 0.5, 0.7, 0.85, 1.0])
samp = geo[geo["class"].isin(TARGET_CLASSES)].copy()
samp["conf_bin"] = pd.cut(samp.confidence, bins)
cols = ["trayecto", "source", "source_file", "t_start", "t_end", "class", "class_name", "confidence"]
parts = []
for _, d in samp.groupby(["class", "conf_bin"], observed=True):
    parts.append(d.sample(min(len(d), N_PER_CELL), random_state=42))
sheet = pd.concat(parts)[cols].copy()
sheet["is_tp"] = ""        # <-- rellenar a mano: 1=correcto, 0=falso positivo
out = "../validation/sampling_sheet.csv"
sheet.to_csv(out, index=False)
print(f"{len(sheet)} clips -> {out}. Escuchar y rellenar 'is_tp'.")
sheet.head()

**Paso 2 — calcular precisión** (se ejecuta cuando la hoja esté rellenada). Si aún no hay etiquetas, esta celda lo avisa y se salta sin romper el notebook.

In [ ]:
lab_path = Path("../validation/labels.csv")
src_path = Path("../validation/sampling_sheet.csv")
use = lab_path if lab_path.exists() else src_path
labels = pd.read_csv(use)
labels = labels[pd.to_numeric(labels["is_tp"], errors="coerce").notna()].copy()
labels["is_tp"] = labels["is_tp"].astype(int)
if labels.empty:
    print("Sin etiquetas todavía. Rellenar is_tp en", src_path, "y re-ejecutar.")
    prec = None
else:
    prec = gu.class_precision(labels)
    display(prec)

In [ ]:
if prec is not None:
    fig, ax = plt.subplots(figsize=(9, 5))
    y = np.arange(len(prec))
    ax.errorbar(prec.precision, y,
                xerr=[prec.precision - prec.ci_lo, prec.ci_hi - prec.precision],
                fmt="o", capsize=5, color="#c0392b")
    ax.set_yticks(y); ax.set_yticklabels(prec["name"]); ax.set_xlim(0, 1)
    ax.set_xlabel("precisión (IC Wilson 95%)")
    ax.set_title(f"Precisión por clase — validación parcial (n={int(prec.n.sum())})")
    plt.tight_layout(); plt.savefig("../outputs/nbA_precision_ci.png", dpi=130); plt.show()
else:
    print("(figura de precisión pendiente de etiquetas)")

## Calibración: confianza → precisión empírica
¿La confianza del modelo predice la corrección? Agrupamos las etiquetas por bin de confianza y comparamos con la diagonal ideal. Define el **umbral operativo por clase**.

In [ ]:
if prec is not None and len(labels) >= 10:
    labels["conf_bin"] = pd.cut(labels.confidence, bins)
    cal = (labels.groupby("conf_bin", observed=True)
                 .agg(emp_prec=("is_tp", "mean"), n=("is_tp", "size"),
                      conf_mid=("confidence", "mean")).reset_index())
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot([0, 1], [0, 1], "--", color="gray", label="ideal")
    ax.scatter(cal.conf_mid, cal.emp_prec, s=cal.n*20, color="#2980b9", zorder=3)
    for _, r in cal.iterrows():
        ax.annotate(f"n={int(r.n)}", (r.conf_mid, r.emp_prec), fontsize=10)
    ax.set_xlabel("confianza media (bin)"); ax.set_ylabel("precisión empírica")
    ax.set_title("Curva de calibración"); ax.legend()
    plt.tight_layout(); plt.savefig("../outputs/nbA_calibration.png", dpi=130); plt.show()
    display(cal)
else:
    print("Calibración pendiente de >=10 etiquetas.")

### Salida del notebook
- Tabla de precisión por clase con IC → `outputs/nbA_precision_ci.png`.
- Umbral operativo por clase (elegir el bin donde la precisión empírica supera el objetivo, p.ej. 0.7). Estos pesos de precisión alimentan el **danger score v2** (NB-04).
- **Limitación declarada:** validación parcial → IC anchos; no se sobrevende precisión.